# TNBC Multi-Drug Combination Hypothesis Pipeline
### Phase-1 runnable scaffold for the MC-ORE / Hybrid-CORE architecture

**What this notebook does (Phase 4 of the roadmap):** takes a patient's altered-gene profile,
scores every candidate drug pair (and top triplets) for predicted synergy, screens them for
overlapping toxicity, generates a plain-language rationale for each, and outputs a **ranked
hypothesis table** — the exact artifact needed to hand to a wet lab for validation
(Section 10/11 of the MC-ORE report).

**Honesty check on what's "real" vs. placeholder right now:**
- The knowledge base (genes → pathways → drugs) below is real and literature-grounded.
- The toxicity flags are real, curated from known drug labels/literature.
- The **synergy scoring function is a documented heuristic placeholder**, not yet a trained
  GNN (SynerGNet/DeepDDS). It combines pathway overlap + mechanistic complementarity + a small
  curated table of literature-confirmed synergistic pairs. Section 4 near the bottom shows
  exactly where to swap in a trained model checkpoint once you complete Phase 2 of the roadmap.
- The "multi-agent rationale" step calls the Anthropic API if a key is available in your
  environment; otherwise it falls back to a template-based explanation so the notebook still
  runs end-to-end with zero API keys or internet access.


In [ ]:
import itertools
import json
import os
from dataclasses import dataclass, field

import pandas as pd

pd.set_option("display.max_colwidth", 80)


## Phase 1 — Knowledge base (KG++ seed)

A small but real knowledge base: gene → pathway membership, gene → approved/investigational
inhibitor(s), and a curated toxicity-flag table. This is deliberately larger than the original
10-gene database and includes BRCA1/2 + PARP inhibitors, which matter specifically for TNBC.

Sources for expansion: DGIdb, OncoKB, KEGG pathway maps — see the roadmap discussion for how to
pull these programmatically instead of hand-curating.


In [ ]:
# gene -> list of pathways it feeds (used for the legacy redundancy score, kept for comparison)
GENE_PATHWAYS = {
    "EGFR":   ["MAPK/ERK", "PI3K/AKT"],
    "ERBB2":  ["MAPK/ERK", "PI3K/AKT"],
    "MET":    ["MAPK/ERK", "PI3K/AKT"],
    "SRC":    ["MAPK/ERK"],
    "ABL1":   ["MAPK/ERK"],
    "FGFR1":  ["MAPK/ERK", "PI3K/AKT"],
    "FGFR2":  ["MAPK/ERK", "PI3K/AKT"],
    "FGFR3":  ["MAPK/ERK", "PI3K/AKT"],
    "ALK":    ["MAPK/ERK", "PI3K/AKT"],
    "KIT":    ["MAPK/ERK", "PI3K/AKT"],
    "PDGFRA": ["MAPK/ERK", "PI3K/AKT"],
    "BRAF":   ["MAPK/ERK"],
    "PIK3CA": ["PI3K/AKT"],
    "AKT1":   ["PI3K/AKT"],
    "MTOR":   ["PI3K/AKT"],
    "JAK2":   ["JAK/STAT"],
    "BRCA1":  ["DNA Damage Repair"],
    "BRCA2":  ["DNA Damage Repair"],
    "CDK4":   ["Cell Cycle"],
    "CDK6":   ["Cell Cycle"],
}

# gene -> approved/investigational inhibitor(s)
GENE_DRUGS = {
    "EGFR":   ["erlotinib", "gefitinib", "afatinib"],
    "ERBB2":  ["lapatinib", "neratinib", "trastuzumab"],
    "MET":    ["crizotinib", "capmatinib"],
    "SRC":    ["dasatinib", "bosutinib"],
    "ABL1":   ["imatinib", "dasatinib", "nilotinib"],
    "FGFR1":  ["erdafitinib", "pemigatinib"],
    "FGFR2":  ["erdafitinib", "pemigatinib"],
    "FGFR3":  ["erdafitinib", "pemigatinib"],
    "ALK":    ["alectinib", "crizotinib"],
    "KIT":    ["imatinib", "sunitinib"],
    "PDGFRA": ["imatinib", "sunitinib"],
    "BRAF":   ["vemurafenib", "dabrafenib"],
    "PIK3CA": ["alpelisib"],
    "AKT1":   ["capivasertib"],
    "MTOR":   ["everolimus", "temsirolimus"],
    "JAK2":   ["ruxolitinib"],
    "BRCA1":  ["olaparib", "talazoparib"],
    "BRCA2":  ["olaparib", "talazoparib"],
    "CDK4":   ["palbociclib", "ribociclib"],
    "CDK6":   ["palbociclib", "ribociclib"],
}

# curated toxicity flags per drug (real, literature/label-derived; not exhaustive)
DRUG_TOXICITY = {
    "erlotinib":    {"hepatotoxic": True,  "cardiotoxic": False, "qt_prolonging": False},
    "gefitinib":    {"hepatotoxic": True,  "cardiotoxic": False, "qt_prolonging": False},
    "afatinib":     {"hepatotoxic": False, "cardiotoxic": False, "qt_prolonging": False},
    "lapatinib":    {"hepatotoxic": True,  "cardiotoxic": True,  "qt_prolonging": True},
    "neratinib":    {"hepatotoxic": True,  "cardiotoxic": True,  "qt_prolonging": False},
    "trastuzumab":  {"hepatotoxic": False, "cardiotoxic": True,  "qt_prolonging": False},
    "crizotinib":   {"hepatotoxic": True,  "cardiotoxic": False, "qt_prolonging": True},
    "capmatinib":   {"hepatotoxic": True,  "cardiotoxic": False, "qt_prolonging": False},
    "dasatinib":    {"hepatotoxic": False, "cardiotoxic": True,  "qt_prolonging": True},
    "bosutinib":    {"hepatotoxic": True,  "cardiotoxic": False, "qt_prolonging": False},
    "imatinib":     {"hepatotoxic": True,  "cardiotoxic": False, "qt_prolonging": False},
    "nilotinib":    {"hepatotoxic": True,  "cardiotoxic": True,  "qt_prolonging": True},
    "erdafitinib":  {"hepatotoxic": False, "cardiotoxic": False, "qt_prolonging": False},
    "pemigatinib":  {"hepatotoxic": True,  "cardiotoxic": False, "qt_prolonging": False},
    "alectinib":    {"hepatotoxic": True,  "cardiotoxic": False, "qt_prolonging": False},
    "sunitinib":    {"hepatotoxic": True,  "cardiotoxic": True,  "qt_prolonging": True},
    "vemurafenib":  {"hepatotoxic": True,  "cardiotoxic": False, "qt_prolonging": True},
    "dabrafenib":   {"hepatotoxic": False, "cardiotoxic": False, "qt_prolonging": False},
    "alpelisib":    {"hepatotoxic": True,  "cardiotoxic": False, "qt_prolonging": False},
    "capivasertib": {"hepatotoxic": True,  "cardiotoxic": False, "qt_prolonging": False},
    "everolimus":   {"hepatotoxic": True,  "cardiotoxic": False, "qt_prolonging": False},
    "temsirolimus": {"hepatotoxic": True,  "cardiotoxic": False, "qt_prolonging": False},
    "ruxolitinib":  {"hepatotoxic": False, "cardiotoxic": False, "qt_prolonging": False},
    "olaparib":     {"hepatotoxic": False, "cardiotoxic": False, "qt_prolonging": False},
    "talazoparib":  {"hepatotoxic": False, "cardiotoxic": False, "qt_prolonging": False},
    "palbociclib":  {"hepatotoxic": True,  "cardiotoxic": False, "qt_prolonging": True},
    "ribociclib":   {"hepatotoxic": True,  "cardiotoxic": False, "qt_prolonging": True},
}

# small curated table of literature-reported synergistic combinations (real, citable pairs)
# used as a boost signal until a trained GNN replaces this lookup entirely
KNOWN_SYNERGY_BOOST = {
    frozenset(["erlotinib", "crizotinib"]):   0.25,   # EGFR+MET bypass-resistance synergy
    frozenset(["olaparib", "alpelisib"]):      0.30,   # PARP + PI3K synergy in BRCA-mutant models
    frozenset(["palbociclib", "alpelisib"]):   0.20,   # CDK4/6 + PI3K synergy
    frozenset(["dasatinib", "everolimus"]):    0.15,   # SRC + mTOR co-targeting
}


## Phase 1b — Ingest a patient profile

Accepts either:
1. A simple CSV with columns `gene, alteration_type, expression_log2fc` (recommended — this is
   what you'd export from a TCGA-BRCA / METABRIC record after variant calling), or
2. A minimal best-effort VCF gene-name extraction (for a real annotated VCF with a `GENE=` or
   `ANN=` INFO tag).

A synthetic demo CSV is generated below so the notebook runs end-to-end without any input file.
Replace `demo_patient.csv` with a real exported profile when you have one.


In [ ]:
DEMO_PATIENT_CSV = "demo_patient.csv"

demo_rows = [
    ("EGFR", "amplification", 2.1),
    ("MET", "amplification", 1.8),
    ("SRC", "missense", 0.6),
    ("BRCA1", "frameshift", -0.2),
    ("PIK3CA", "missense", 1.1),
    ("CDK4", "amplification", 1.4),
]
pd.DataFrame(demo_rows, columns=["gene", "alteration_type", "expression_log2fc"]).to_csv(
    DEMO_PATIENT_CSV, index=False
)

def load_patient_profile(path):
    """Load a patient's altered-gene profile from CSV (or best-effort from a VCF)."""
    if path.lower().endswith(".vcf"):
        genes = set()
        with open(path) as f:
            for line in f:
                if line.startswith("#"):
                    continue
                info = line.strip().split("\t")
                if len(info) < 8:
                    continue
                info_field = info[7]
                for tag in ("GENE=", "ANN="):
                    if tag in info_field:
                        chunk = info_field.split(tag)[1].split(";")[0]
                        genes.add(chunk.split("|")[0].strip())
        df = pd.DataFrame({"gene": sorted(genes), "alteration_type": "unspecified",
                            "expression_log2fc": 0.0})
    else:
        df = pd.read_csv(path)

    # keep only genes present in our knowledge base
    df = df[df["gene"].isin(GENE_PATHWAYS)].reset_index(drop=True)
    return df

patient_df = load_patient_profile(DEMO_PATIENT_CSV)
patient_df


## Phase 1c — Legacy pathway-redundancy score (kept for comparison to Section 5/6 of the report)

This is the *original* rule-based scoring method, retained so you can show your professor the
before/after difference directly, exactly as the comparison table in Section 5 describes.


In [ ]:
import math

def pathway_redundancy_score(genes, pathway):
    """Sigmoid-like redundancy score from shared-pathway gene count (legacy method)."""
    n = sum(1 for g in genes if pathway in GENE_PATHWAYS.get(g, []))
    if n == 0:
        return 0.0
    return 1 / (1 + math.exp(-(n - 1)))

altered_genes = patient_df["gene"].tolist()
pathways = sorted({p for g in altered_genes for p in GENE_PATHWAYS.get(g, [])})
legacy_scores = {p: round(pathway_redundancy_score(altered_genes, p), 3) for p in pathways}
legacy_scores


## Phase 2 — Synergy prediction layer (MM-SynergyNet interface)

`SynergyModel.predict_pair()` is the function to replace once you've trained a real GNN
(Phase 2 of the roadmap). The interface is deliberately minimal — `(drug_a, drug_b, context) ->
float score` — so swapping in a checkpoint from SynerGNet/DeepDDS later requires touching only
this one class, nothing downstream.

**Current heuristic** (documented, not hidden): pathway overlap (shared pathway between the two
genes) + mechanistic complementarity bonus (RTK + intracellular kinase pairings score higher
than same-class pairs) + the curated literature-synergy lookup above.


In [ ]:
RTK_GENES = {"EGFR", "ERBB2", "MET", "FGFR1", "FGFR2", "FGFR3", "ALK", "KIT", "PDGFRA"}
INTRACELLULAR_GENES = {"SRC", "ABL1", "BRAF", "PIK3CA", "AKT1", "MTOR", "JAK2", "CDK4", "CDK6"}

class SynergyModel:
    """
    Placeholder synergy-scoring interface.

    TO SWAP IN A TRAINED GNN (SynerGNet / DeepDDS style):
        1. Train per the Phase 2 roadmap step, using DrugCombDB / AZ-DREAM as labels.
        2. Save a checkpoint and load it in __init__ below.
        3. Replace the body of predict_pair() with a forward pass over the drug-pair's
           molecular graph + PPI-neighbourhood features; keep the same (float, 0-1) return type
           so nothing downstream needs to change.
    """

    def __init__(self, model_checkpoint_path=None):
        self.model = None
        if model_checkpoint_path:
            # e.g. self.model = torch.load(model_checkpoint_path)
            raise NotImplementedError("Plug in your trained GNN loader here.")

    def predict_pair(self, gene_a, gene_b, drug_a, drug_b):
        if self.model is not None:
            raise NotImplementedError("Real model inference path not yet implemented.")

        # --- heuristic placeholder ---
        pathways_a = set(GENE_PATHWAYS.get(gene_a, []))
        pathways_b = set(GENE_PATHWAYS.get(gene_b, []))
        overlap = len(pathways_a & pathways_b)
        overlap_score = min(overlap * 0.15, 0.4)

        classes = {gene_a in RTK_GENES, gene_a in INTRACELLULAR_GENES,
                   gene_b in RTK_GENES, gene_b in INTRACELLULAR_GENES}
        complementary = (gene_a in RTK_GENES and gene_b in INTRACELLULAR_GENES) or \
                         (gene_b in RTK_GENES and gene_a in INTRACELLULAR_GENES)
        complementarity_score = 0.25 if complementary else 0.1

        boost = KNOWN_SYNERGY_BOOST.get(frozenset([drug_a, drug_b]), 0.0)

        score = min(overlap_score + complementarity_score + boost, 1.0)
        return round(score, 3)

synergy_model = SynergyModel()


## Phase 2b — Safety / toxicity screening (Safety Agent equivalent)

Flags any candidate pair where both drugs share a toxicity category. This is a direct,
checkable stand-in for the Safety Agent described in Section 4.3 of the report — replace the
lookup table with real assay results once Section 11's wet-lab data starts coming back
(Stage 4 of the translational loop).


In [ ]:
def safety_screen(drug_a, drug_b):
    tox_a = DRUG_TOXICITY.get(drug_a, {})
    tox_b = DRUG_TOXICITY.get(drug_b, {})
    shared_flags = [cat for cat in tox_a if tox_a.get(cat) and tox_b.get(cat)]
    return {
        "flagged": len(shared_flags) > 0,
        "shared_toxicity_categories": shared_flags,
    }


## Phase 3 — Multi-agent rationale (optional LLM, safe fallback)

Calls the Anthropic API to generate a short Genomics/Pharmacology/Safety/Chair-style rationale
*if* `ANTHROPIC_API_KEY` is set in your environment. If not, falls back to a template-based
sentence so the notebook still runs fully offline.


In [ ]:
def generate_rationale(gene_a, gene_b, drug_a, drug_b, synergy_score, safety_result):
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    safety_note = (
        f"Safety Agent flags overlapping {', '.join(safety_result['shared_toxicity_categories'])} risk."
        if safety_result["flagged"] else
        "Safety Agent finds no overlapping toxicity flags in the curated table."
    )
    template = (
        f"{gene_a}+{gene_b} co-alteration suggests combining {drug_a} and {drug_b} "
        f"(predicted synergy {synergy_score}). {safety_note}"
    )

    if not api_key:
        return template

    try:
        import anthropic
        client = anthropic.Anthropic(api_key=api_key)
        prompt = (
            f"Act as a four-agent oncology reasoning panel (Genomics, Pharmacology/Synergy, "
            f"Safety, Chair). Patient has {gene_a} and {gene_b} alterations. Candidate combo: "
            f"{drug_a} + {drug_b}, predicted synergy score {synergy_score} (0-1 scale). "
            f"{safety_note} Give a 2-3 sentence Chair-agent-style consensus rationale, "
            f"noting any disagreement between agents if the safety flag is raised."
        )
        resp = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=200,
            messages=[{"role": "user", "content": prompt}],
        )
        return resp.content[0].text.strip()
    except Exception as e:
        return template + f" (LLM call unavailable: {e})"


## Phase 4 — Run the full pipeline: build the ranked hypothesis table

This is the deliverable table for your professor: every candidate pair (from the patient's
altered genes) scored for synergy, screened for safety, and given a plain-language rationale —
directly matching the Phase 4 output spec from the roadmap and feeding Stage 1 of the wet-lab
loop (Section 10 of the report).


In [ ]:
def run_pipeline(patient_df, top_n=15, include_triplets_for_top_k=3):
    genes = patient_df["gene"].tolist()
    rows = []

    # pairwise candidates: every gene pair where each has at least one known drug
    for gene_a, gene_b in itertools.combinations(genes, 2):
        for drug_a in GENE_DRUGS.get(gene_a, []):
            for drug_b in GENE_DRUGS.get(gene_b, []):
                if drug_a == drug_b:
                    continue
                score = synergy_model.predict_pair(gene_a, gene_b, drug_a, drug_b)
                safety = safety_screen(drug_a, drug_b)
                rows.append({
                    "combination": f"{drug_a} + {drug_b}",
                    "genes": f"{gene_a}+{gene_b}",
                    "predicted_synergy_score": score,
                    "safety_flag": safety["flagged"],
                    "toxicity_overlap": ", ".join(safety["shared_toxicity_categories"]) or "none",
                    "n_drugs": 2,
                })

    df = pd.DataFrame(rows).drop_duplicates(subset=["combination"])
    df = df.sort_values("predicted_synergy_score", ascending=False).reset_index(drop=True)
    df = df.head(top_n).copy()

    # rationale only for the final shortlist (keeps LLM calls, if any, cheap)
    rationales = []
    for _, r in df.iterrows():
        gene_a, gene_b = r["genes"].split("+")
        drug_a, drug_b = r["combination"].split(" + ")
        safety = {"flagged": r["safety_flag"],
                  "shared_toxicity_categories": [] if r["toxicity_overlap"] == "none"
                                                 else r["toxicity_overlap"].split(", ")}
        rationales.append(generate_rationale(gene_a, gene_b, drug_a, drug_b,
                                              r["predicted_synergy_score"], safety))
    df["rationale"] = rationales

    # recommended pilot = highest synergy with no safety flag
    unflagged = df[~df["safety_flag"]]
    df["recommended_first_pilot"] = False
    if len(unflagged):
        df.loc[unflagged.index[0], "recommended_first_pilot"] = True

    return df

hypothesis_table = run_pipeline(patient_df, top_n=15)
hypothesis_table


In [ ]:
# export for your professor / the wet lab (Stage 1 of the translational loop)
output_path = "tnbc_ranked_drug_combinations.csv"
hypothesis_table.to_csv(output_path, index=False)
print(f"Wrote {len(hypothesis_table)} ranked candidates to {output_path}")

print()
recommended = hypothesis_table[hypothesis_table["recommended_first_pilot"]]
if len(recommended):
    print("Recommended first wet-lab pilot combination:")
    print(recommended[["combination", "genes", "predicted_synergy_score", "rationale"]].to_string(index=False))


## Next steps to make this production-grade

1. **Replace the demo CSV** with a real exported TCGA-BRCA/METABRIC or institutional patient
   profile — `load_patient_profile()` already supports both CSV and best-effort VCF parsing.
2. **Train a real synergy model** (Phase 2 of the roadmap): clone SynerGNet or DeepDDS from
   GitHub, fine-tune on DrugCombDB, and load the checkpoint into `SynergyModel.__init__`. The
   rest of the pipeline needs zero changes.
3. **Expand the knowledge base**: pull DGIdb/OncoKB programmatically instead of hand-curating
   `GENE_DRUGS`/`GENE_PATHWAYS`, so new genes/drugs are added automatically (this becomes KG++).
4. **Wire in real RAG evidence**: replace the template rationale with retrieved PubMed/
   ClinicalTrials.gov abstracts per candidate, as described in Section 4.3 of the report.
5. **Send `tnbc_ranked_drug_combinations.csv` + the recommended pilot to your professor** —
   this is the exact input Stage 1 ("Define model-derived hypotheses") of the wet-lab loop
   needs, and maps directly onto the EGFR+MET experimental design already sketched in
   Section 11 of the report.
